# Collab de Referência do Capítulo 7
## Técnicas avançadas e ferramentas de geração de texto

<font color='#EE3333'>Este é o Collab de respostas, todos os espaços estão preenchidos</font>

## Instalação e importação das bibliotecas
Execute essa célula para instalar todas as dependências necessárias

In [ ]:
# Execute para instalar as bibliotecas necessárias
!pip install langchain
!pip install langchain-groq
!pip install llama-cpp-python
!pip install langchain-community
!pip install duckduckgo-search
!pip install ddgs
!pip install

# Baixando o modelo Phi-3
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

# Importando bibliotecas
# PS.: tive que mudar as importações, as do livro estavam deprecadas e mudei a llm da OPENAI para GROQ (llama) para não pagar por token
import os
from langchain_groq import ChatGroq
from langchain_community.llms import LlamaCpp
from langchain_community.agent_toolkits.load_tools import load_tools, Tool
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.prompts import PromptTemplate
from langchain_core.callbacks import BaseCallbackHandler

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 16.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4503299 sha256=f0e306a82dae461f750ef5325d5b2931c2b0d9aeb873d184ce0865a03518e1b1
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python
ERROR: You must give at least one requirement to install (see "pip help install")
--2026-03-09 02:29:03--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.17, ...
Connecting to huggingface.

## Importando Modelo Quantizado

### Questão 1. Importando Modelo

Como visto em sala de aula, é necessário alguns passos para que possamos ter uma conversa real com a LLM, primeiramente precisamos importar o modelo que será usado,e seguindo a forma de modelo comentado,ele será quantizado em 16bits para ter mais velocidade,usaremos o Phi-3 na variante gguf ( Phi-3-mini-4k-instruct-fp16.gguf).

In [ ]:
# Criando o Modelo
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    temperature=0,
    n_ctx=2048,
    seed=42,
    verbose=False
)

llama_context: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64
llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


## Chains

Agora nós iremos trabalhar com as chains propriamente ditas.

Vamos iniciar com uma chain simples.

Nós já temos a llm configurada no modelo anterior, agora devemos criar um template para o prompt que será usado juntamente com a llm.

In [ ]:
# Definindo o Template do Prompt
template = """<|user|>
  {input_prompt}<|end|>
  <|assistant|>"""

# Configurando o prompt
prompt = PromptTemplate(
  template=template,
  input_variables=["input_prompt"]
)

'''
  Estamos trabalhando com uma chain básica,
  vimos na apresentação que a chain usa de dois elementos:

  Um componente (estrutura do prompt)
  A LLM

  Insira esses elementos na linha abaixo
'''

basic_chain = prompt | llm

In [ ]:
'''
  Ultilize a função invoke da chain que criamos.
'''

# Usando a chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

## Multiple Chains

O que você acabou de fazer foi criar um template de prompt, estruturar a chain e realizar um input para o modelo, mas tudo isso para uma chain simples.

Agora veremos o real potencial das chains

Vamos criar um caso de chains multiplas para o exemplo de geração de história que foi usado no livro.

Um prompt que gere uma história com um título, um personagem principal e um sumário da história.

In [ ]:
from langchain_classic.chains import LLMChain

# Assim como na chain simples, devemos primeiro criar o template que será usado
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""

# Devemos agora indicar as variaveis que definimos dentro do template
title_prompt = PromptTemplate(template=template, input_variables=["summary"])

# Cria uma chain para a geração do título
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

In [ ]:
# Cuidado ao executar esta célula
# Pode demandar muito tempo de execução
title.invoke({"summary": "a girl that lost her mother"})

Dessa forma ainda estamos usando uma chain simples.
Vamos complicar um pouco.

In [ ]:
# Criando uma chain para a geração do personagem principal
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""

character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)

# Defina o que nós queremos como saída desta chain
# Dica: apenas uma palavra
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [ ]:
# Criando um template com os três elementos
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""

# Use mais uma vez as variáveis que foram definidas no template
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)

# Agora utilize a função que cria uma chain
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [ ]:
'''
  Nas células anteriores nós criamos 3 chains separadas
  Geração do título
  Geração do personagem principal
  Geração da história (a descrição da história)

  Agora una esses três elementos para cria a chain multipla
'''

llm_chain = title | character | story

In [ ]:
# Cuidado ao executar esta célula
# Demanda muito tempo de execução
llm_chain.invoke("a girl that lost her mother")

## Colocando Memória na LLM

### Questão 2. Memorização Total

Uma das formas mais claras de fazer com que a LLM possa lembrar do que foi dito anteriormente na conversa, é simplesmente passando todo histórico da conversa como parte do User Prompt, para isso podemos usar o ConversationBufferMemory.

Importe-o e preencha o código com a classe referida:

In [ ]:
from langchain_classic.memory import ConversationBufferMemory

# Criando o template que usaremos para os próximos dois modelos de memória
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

# Define the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Combine o LLM, o Prompt e a Memória
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [ ]:
# Faça prefencialmente perguntas matemáticas básicas (como quanto é 1 + 1)

from pprint import pprint

pergunta = input()

while pergunta != '0':
  resposta = llm_chain.invoke({"input_prompt": pergunta})

  pprint(resposta)

  pergunta = input()

### Questão 3. Memorização Parcial

O método anterior apresenta um problema claro, aumenta a quantidade de tokens sendo gerados em cada invoke, o que pode acabar fazendo com que a LLM demore mais para responder ou simplesmente estoure o limite de tokens permitido, uma solução para este problema é salvar apenas as últimas k interações com a LLM, usando o ConversationBufferMemoryWindow.

Importe-o e preencha os espaços faltantes, também defina a quantidade de interações que devem ser salvas (recomendado = 2):

In [ ]:
from langchain_classic.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [ ]:
# Faça prefencialmente perguntas matemáticas básicas (como quanto é 1 + 1)

from pprint import pprint

pergunta = input()

while pergunta != '0':
  resposta = llm_chain.invoke({"input_prompt": pergunta})

  pprint(resposta)

  pergunta = input()

### Questão 4. Sumarização das Interações

Embora o método anterior solucione o principal problema do ConversationBuffer, está longe de ser a melhor opção, devido ao fato de apenas lembrar das últimas interações com o usuário, se tornando inviável para conversas mais longas que exijam muitos detalhes, tentando solucionar ambos os problemas podemos recorrer a uma sumarização da conversa esta sendo feita pela própria llm, utilizando o ConversationSummaryMemory.

Importe-o e preencha os espaços faltantes, adicione a variável referente a LLM no campo desejado:

In [ ]:
from langchain_classic.memory import ConversationSummaryMemory

# Prompt Template
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

# Defina o tipo de memória
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

Teste:

In [ ]:
# Faça prefencialmente perguntas matemáticas básicas (como quanto é 1 + 1)

from pprint import pprint

pergunta = input()

while pergunta != '0':
  resposta = llm_chain.invoke({"input_prompt": pergunta})

  pprint(resposta)

  pergunta = input()

## Integração de LLMs com ferramentas externas usando LangChain

### Questão 5. Início do código
Para conectar a LLM às ferramentas (pesquisa e calculadora) usando o LangChain e o framework ReAct <font color='grey'>(não confundir com React do JS)</font>
primeiro importamos o modelo.

Vamos usar o modelo "llama-3.3-70b-versatile", mas graças a versatilidade do LangChain, é possível importar qualquer modelo nessa etapa e a "corrente", bem como as ferramentas não serão afetadas. Caso o llama não esteja disponível, por exemplo, uma alternativa é o modelo "qwen/qwen3-32b".

In [ ]:
# primeiro, carregamos a LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="",
    temperature=0
)

A classe abaixo não tem exercícios, ela é um handler para formatar o output da LLM para compreendermos melhor o processamento.

In [ ]:
# Extra: essa classe não está no livro, serve apenas para formatar a saída
#        para facilitar a compreensão.
# Não há exercícios aqui

class PrettyReActHandler(BaseCallbackHandler):
    def on_agent_action(self, action, **kwargs):
        log = action.log
        if "Thought:" in log:
            thought = log.split("Thought:")[-1].split("Action:")[0].strip()
            print("\nThought:", thought)
        print("\nAction:", action.tool)
        print("Action Input:", action.tool_input)

    def on_tool_end(self, output, **kwargs):
        print("Observation:", output, "\n")

    def on_agent_finish(self, finish, **kwargs):
        print("\nFinal Answer:", finish.return_values["output"])
        print()

### Questão 6. Definindo template de prompt do ReAct
O template indica para a LLM que deve serguir esses passos de processamento e uso das ferramentas para resolver a questão do usuário.
Depois, precisamos associar essa string com o template ao LangChain, esse modelo de prompt será um dos elos da nossa corrente, enviada para o modelo.

In [ ]:
# Definindo o passo a passo base para o modelo
react_template = """
  Answer the following questions as best you can. You have
  access to the following tools:
  {tools}
  Do not output <think> tags.
  Use the following format:
  Question: the input question you must answer
  Thought: you should always think about what to do
  Action: the action to take, should be one of [{tool_names}]
  Action Input: the input to the action
  Observation: the result of the action
  ... (this Thought/Action/Action Input/Observation can repeat N times)
  Thought: I now know the final answer
  Final Answer: the final answer to the original input question
  Begin!
  Question: {input}
  Thought:{agent_scratchpad}
"""

# criando o prompt para ser usado pelo LangChain
prompt = PromptTemplate(
  template=react_template,
  input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

### Questão 7. Criação das ferramentas para serem usadas pelo modelo
Importamos a ferramenta de pesquisa do duckduckgo e a carregamos llm-math, ambos poderão ser acessadas pela LLM para realizar tarefas. A classe DuckDuckGoSearchResults é externa, por isso primeiro criamos uma Tool com a função de pesquisa, depois adicionamos a ferramenta nova junto à ferramenta llm-math.

In [ ]:
# Criando ferramenta para o agente, importando motor de pesquisa do duck duck go
search = DuckDuckGoSearchResults()
search_tool = Tool(
  name="duckduck",
  description="A web search engine. Use this to as a search engine for general queries.",
  func=search.run,
)

# Preparando ferramentas
tools = load_tools(["llm-math"], llm=llm)
tools.append(search_tool)

### Questão 8. Construção do agente ReAct e teste!
Construímos o agente com a função create_react_agent passando as ferramentas, llm e o template de prompt que criamos anteriormente, depois criamos um executor para o nosso agente, e por fim, usamos a função invoke para enviar nosso input e executar a corrente ReAct/LangChain.

In [ ]:
# Construindo o agente ReAct
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
  agent=agent, tools=tools, verbose=False, callbacks=[PrettyReActHandler()], handle_parsing_errors=True
)

# Digite aqui o seu prompt:
input_usuario = "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.86 EUR for 1 USD."

agent_executor.invoke(
  {
    "input": input_usuario
  }
)


Action: duckduck
Action Input: current price of MacBook Pro in USD

Final Answer: The current price of a MacBook Pro starts at $2,199 USD. Converting this to EUR with an exchange rate of 0.86, the cost would be €1,891.14.



{'input': 'What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.86 EUR for 1 USD.',
 'output': 'The current price of a MacBook Pro starts at $2,199 USD. Converting this to EUR with an exchange rate of 0.86, the cost would be €1,891.14.'}

# Referência


<h1>Hands-On Large Language Models</h1>
<i>Language Understanding and Generation.</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>

---

[Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>